In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# panel
panel_df = pd.read_parquet('../data/processed/panel_df.parquet')

## propensity score matching

### prep dataset

In [ ]:
df = panel_df.sort_values(['firm_id', 'year']).reset_index(drop=True)
df['firm_age_t'] = df['year'] - df['incorp_year']

covariates = [
    'log_rev', 'log_ass', 'log_empl', 'ebit_marg', 
    'log_lev', 'log_rd_int', 'pat_lag1', 'rev_growth', 
    'hhi', 'nace_pat_int','ctry_code_freq', 'nace_code_freq'
]
df[covariates] = df[covariates].replace([np.inf, -np.inf], np.nan)

for v in covariates:
    df[v+'_lag'] = df.groupby('firm_id')[v].shift(1)

# drop years without lagged covariates
df = df.dropna(subset=[v+'_lag' for v in covariates])
df = df.reset_index(drop=True)

lagged_covs = [v + '_lag' for v in covariates]

X = df[lagged_covs].copy()
y = df['treat_t']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

ps_model = LogisticRegression(max_iter=2000)
ps_model.fit(X_scaled, y)

df['ps'] = ps_model.predict_proba(X_scaled)[:, 1]

In [ ]:
matched_rows = []

for year in sorted(df['year'].unique()):

    # firm-years with treatment this year
    df_treated = df[(df['year'] == year) & (df['treat_t'] == 1)]
    df_controls = df[(df['year'] == year) & (df['treat_t'] == 0)]

    if df_treated.empty or df_controls.empty:
        continue

    X_t = scaler.transform(df_treated[lagged_covs])
    X_c = scaler.transform(df_controls[lagged_covs])

    nn = NearestNeighbors(n_neighbors=3, metric='euclidean')  # 1:3 matching
    nn.fit(X_c)

    distances, indices = nn.kneighbors(X_t)

    for i, idx_list in enumerate(indices):
        treated_row = df_treated.iloc[i]
        matched_controls = df_controls.iloc[idx_list]
        matched_rows.append(pd.concat([treated_row.to_frame().T, matched_controls], axis=0))

matched_df = pd.concat(matched_rows, ignore_index=True)

print("Matched dataset shape:", matched_df.shape)
print("Treated firm-years:", df[df.treat_t == 1].shape[0])
print("Matched firm-years:", matched_df.shape[0])


In [ ]:
num_treated_firms = matched_df.loc[matched_df['treat']==1, 'firm_id'].nunique()
num_control_firms = matched_df.loc[matched_df['treat']==0, 'firm_id'].nunique()
print(num_treated_firms)
print(num_control_firms)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# separate treated vs control (firm-years) based on treat_t
treated = df[df['treat_t'] == 1]['ps']
control = df[df['treat_t'] == 0]['ps']

plt.figure(figsize=(10,5))

plt.hist(control, bins=20, alpha=0.5, density=True, label='Control', color='blue')
plt.hist(treated, bins=20, alpha=0.5, density=True, label='Treated', color='orange')

plt.xlabel('Propensity Score')
plt.ylabel('Density')
plt.title('Density Histogram of Propensity Scores: Treated vs Control')
plt.legend()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# separate treated vs control (firm-years) based on treat_t
treated = matched_df[matched_df['treat_t'] == 1]['ps']
control = matched_df[matched_df['treat_t'] == 0]['ps']

plt.figure(figsize=(10,5))

plt.hist(control, bins=20, alpha=0.5, density=True, label='Control', color='blue')
plt.hist(treated, bins=20, alpha=0.5, density=True, label='Treated', color='orange')

plt.xlabel('Propensity Score')
plt.ylabel('Density')
plt.title('Density Histogram of Propensity Scores: Treated vs Control')
plt.legend()
plt.show()

## nn matching

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm

df = panel_df.copy()

covariates = [
    'log_rev', 'log_ass', 'log_empl', 'ebit_marg',
    'log_lev', 'log_rd_int', 'pat_lag1', 'rev_growth',
    'hhi', 'nace_pat_int','ctry_code_freq', 'nace_code_freq'
]

K = 5   # number of controls per treated firm

# ------------------------------
# 1. Identify treated + control firms
# ------------------------------
treated_firms = df.loc[df['first_treat'].notna(), 'firm_id'].unique()
control_firms = df.loc[df['first_treat'].isna(), 'firm_id'].unique()

# ------------------------------
# 2. Precompute ALL yearly averages for ALL firms
# This avoids recomputing averages millions of times.
# ------------------------------

print("Precomputing yearly averages for all firms...")

# Group by firm-year first
firm_year_avg = df.groupby(['firm_id', 'year'])[covariates].mean()

# Cumulative sums for fast "year < cutoff" averaging
cum_sum = firm_year_avg.groupby(level=0).cumsum()
cum_count = df.groupby(['firm_id', 'year']).size().groupby(level=0).cumsum()

# ------------------------------
# Function: fast pre-treatment average using cumulative sums
# ------------------------------
def fast_pre_avg(fid, cutoff):
    # Find all years < cutoff for this firm
    mask = (cum_sum.index.get_level_values('firm_id') == fid) & \
           (cum_sum.index.get_level_values('year') < cutoff)
    if not mask.any():
        return None

    sum_vals = cum_sum.loc[mask].iloc[-1].values
    count = cum_count.loc[mask].iloc[-1]

    avg = sum_vals / count
    return avg

# ------------------------------
# 3. Matching with progress bar + imputation
# ------------------------------
matched_controls = {}

print("Starting matching...")
for f in tqdm(treated_firms, desc="Matching treated firms"):

    Ti = df.loc[df.firm_id == f, 'first_treat'].iloc[0]

    treated_vec = fast_pre_avg(f, Ti)
    if treated_vec is None:
        continue
    treated_vec = treated_vec.reshape(1, -1)

    # Collect control firm vectors
    control_vecs = []
    control_ids = []

    for c in control_firms:
        cv = fast_pre_avg(c, Ti)
        if cv is None:
            continue
        control_vecs.append(cv)
        control_ids.append(c)

    if len(control_vecs) == 0:
        continue

    control_mat = np.vstack(control_vecs)

    # --------------------------
    # Clean missing / inf values
    # --------------------------
    control_mat = np.where(np.isfinite(control_mat), control_mat, np.nan)
    treated_vec = np.where(np.isfinite(treated_vec), treated_vec, np.nan)

    # median impute
    med = np.nanmedian(control_mat, axis=0)
    for col in range(control_mat.shape[1]):
        control_mat[np.isnan(control_mat[:, col]), col] = med[col]
        if np.isnan(treated_vec[0, col]):
            treated_vec[0, col] = med[col]

    # --------------------------
    # Nearest neighbor match
    # --------------------------
    nn = NearestNeighbors(n_neighbors=min(K, len(control_ids)),
                          metric='euclidean')
    nn.fit(control_mat)
    _, idx = nn.kneighbors(treated_vec)

    matched_controls[f] = [control_ids[i] for i in idx[0]]


# ------------------------------
# 4. Build restricted sample
# ------------------------------
unique_matched_controls = sorted(set(
    c for sublist in matched_controls.values() for c in sublist
))

restricted_ids = list(treated_firms) + unique_matched_controls
restricted_df = df[df.firm_id.isin(restricted_ids)].copy().reset_index(drop=True)

print("\n=== SUMMARY ===")
print("Treated firms:", len(treated_firms))
print("Matched control firms:", len(unique_matched_controls))
print("Restricted sample shape:", restricted_df.shape)
print("Unique firms:", restricted_df['firm_id'].nunique())

In [ ]:
restricted_df.to_parquet('../data/processed/restricted_df.parquet', engine='fastparquet', index=False)
restricted_df.to_csv('../data/processed/restricted_df.csv', index=False)

## staggered did

In [52]:
df = pd.read_parquet('../data/processed/restricted_df.parquet')